# Data preparation 

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from pathlib import Path

df = pd.read_excel("../data/preparation/asics.xlsx")
print("Dataset size:", df.shape)
print(df.columns.tolist())
print(df.head())

Dataset size: (206, 17)
['ID', 'Text', 'Text_type', 'Leadership_context', 'Information_valence', 'Communication_type', 'Source_type', 'Company', 'Country', 'Year', 'Covid', 'Communicator_role', 'Communicator_name', 'Comment', 'year', 'Coding_ambiguity', 'Alternative_context']
            ID                                               Text  Text_type  \
0  JPN20A00-00  Ongoing Digital, Personal and Sustainable Adva...     title    
1  JPN20A01-00                         Anima Sana in Corpore Sano      title   
2  JPN20A01-01      Creating Value Based on Human-Centric Science      title   
3  JPN20A01-02  ASICS is a company that began by making sports...  body_text   
4  JPN20A01-03  We believe in the importance of continuing to ...  body_text   

  Leadership_context Information_valence Communication_type  \
0              OTHER             neutral           message    
1              OTHER             neutral           message    
2              STRAT            positive           me

In [3]:
required_columns = [
    "ID",
    "Text",
    "Company",
    "Leadership_context",
    "Information_valence"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("Missing IDs:", df["ID"].isna().sum())
print("Missing company names:", df["Company"].isna().sum())

print("\nLeadership context categories:")
print(df["Leadership_context"].value_counts(dropna=False))

print("\nInformation valence categories:")
print(df["Information_valence"].value_counts(dropna=False))

Missing IDs: 0
Missing company names: 0

Leadership context categories:
Leadership_context
STRAT     56
STAKE     35
PERF      30
OTHER     28
PEOPLE    22
MARKET    20
CHALL     10
STRAT      2
STAKE      2
StRAT      1
Name: count, dtype: int64

Information valence categories:
Information_valence
neutral      132
positive      49
neutral       10
negative       9
positive       5
NaN            1
Name: count, dtype: int64


In [ ]:
# Use a fixed random seed so the sample is reproducible
RANDOM_SEED = 42
SAMPLE_FRACTION = 0.20

# Remove rows without a company name or unique ID
df_clean = df.dropna(subset=["ID", "Company"]).copy()

if df_clean["ID"].duplicated().any():
    raise ValueError(
        "ID values are not unique. Check duplicates before sampling."
    )

# Sample approximately 20% from each company
sample = (
    df_clean.groupby("Company", group_keys=False)
    .sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED)
    .sort_values(["Company", "ID"])
    .reset_index(drop=True)
)

print("Total records:", len(df_clean))
print("Sample records:", len(sample))
print("Actual sample percentage:",
      round(len(sample) / len(df_clean) * 100, 2), "%")

print("\nSample size by company:")
print(sample.groupby("Company").size())

In [ ]:
sampling_summary = pd.concat(
    [
        df_clean.groupby("Company").size().rename("Total_records"),
        sample.groupby("Company").size().rename("Sample_records")
    ],
    axis=1
).fillna(0)

sampling_summary["Sample_percentage"] = (
    sampling_summary["Sample_records"]
    / sampling_summary["Total_records"] * 100
).round(2)

print(sampling_summary)

In [ ]:
# Include only information needed for independent classification.
# Adapt these fields to your predefined coding guidelines.
candidate_columns = [
    "ID",
    "Text",
    "Text_type",
    "Communication_type",
    "Source_type",
    "Company",
    "Country",
    "Year",
    "Covid",
    "Communicator_role",
    "Communicator_name"
]

# Retain only columns that exist in your dataset
ai_columns = [
    col for col in candidate_columns if col in sample.columns
]

# AI coding sheet: no researcher classification columns
ai_sheet = sample[ai_columns].copy()

# Researcher reference sheet: original manual classifications
reference = sample[
    ["ID", "Leadership_context", "Information_valence"]
].copy()

# Save the two files separately
ai_sheet.to_excel("AI_coding_sample.xlsx", index=False)
reference.to_excel("researcher_reference.xlsx", index=False)

print("AI sample saved:", len(ai_sheet), "records")
print("Reference file saved separately.")

You are acting as an independent second coder for an academic content analysis.

Your task is to classify each record using the predefined coding framework provided below. You must code independently, without access to the researcher's original classifications.

Coding guidelines

[Insert the complete original coding guidelines for Leadership Context and Information Valence here, including category definitions, inclusion/exclusion rules, decision rules, and examples.]

Instructions

Classify every record according to the supplied coding guidelines. Do not invent new categories.
For Leadership_context_AI, apply the Leadership Context coding framework.
For Information_valence_AI, apply the Information Valence coding framework.
Base your decisions on the supplied text and only the contextual information permitted by the coding guidelines.
Do not infer or reproduce any original researcher classifications.
Do not omit records. Preserve each record's original ID exactly.
If a record is genuinely ambiguous, apply the framework's decision rules. If a decision cannot be made, flag it for review rather than inventing a label.
Return a table containing exactly these columns: ID, Leadership_context_AI, and Information_valence_AI.
Use the original category labels exactly as defined in the coding framework.
Do not change the dataset or its original IDs.
Return the completed classifications as an Excel or CSV file. Do not include the researcher's original coding labels in the output.

In [ ]:
ai_results = pd.read_excel("AI_coding_results.xlsx")

print(ai_results.head())
print(ai_results.columns.tolist())

In [ ]:
# Check the AI output
expected_ai_columns = [
    "ID",
    "Leadership_context_AI",
    "Information_valence_AI"
]

missing = [
    col for col in expected_ai_columns
    if col not in ai_results.columns
]

if missing:
    raise ValueError(f"Missing AI columns: {missing}")

if ai_results["ID"].duplicated().any():
    raise ValueError("The AI results contain duplicate IDs.")

if set(ai_results["ID"]) != set(reference["ID"]):
    raise ValueError(
        "AI IDs do not match the sampled reference IDs. "
        "Check for missing or extra records."
    )

# Merge researcher and AI classifications
comparison = reference.merge(
    ai_results[expected_ai_columns],
    on="ID",
    how="inner",
    validate="one_to_one"
)

print("Records matched:", len(comparison))
print(comparison.head())

In [ ]:
def calculate_reliability(data, researcher_col, ai_col):
    # Compare only records with valid classifications from both coders
    valid = data[[researcher_col, ai_col]].dropna()

    researcher_labels = valid[researcher_col].astype(str).str.strip()
    ai_labels = valid[ai_col].astype(str).str.strip()

    # Remove labels that become empty after trimming whitespace
    nonempty = (
        researcher_labels.ne("") & ai_labels.ne("")
    )

    researcher_labels = researcher_labels[nonempty]
    ai_labels = ai_labels[nonempty]

    n = len(researcher_labels)

    if n == 0:
        return {
            "N": 0,
            "Exact_agreement_pct": np.nan,
            "Cohen_kappa": np.nan
        }

    agreement = (
        researcher_labels.to_numpy() == ai_labels.to_numpy()
    ).mean() * 100

    kappa = cohen_kappa_score(
        researcher_labels,
        ai_labels
    )

    return {
        "N": n,
        "Exact_agreement_pct": round(agreement, 2),
        "Cohen_kappa": round(kappa, 4)
            if np.isfinite(kappa) else np.nan
    }


leadership_result = calculate_reliability(
    comparison,
    "Leadership_context",
    "Leadership_context_AI"
)

valence_result = calculate_reliability(
    comparison,
    "Information_valence",
    "Information_valence_AI"
)

results = pd.DataFrame(
    [
        {
            "Variable": "Leadership Context",
            **leadership_result
        },
        {
            "Variable": "Information Valence",
            **valence_result
        }
    ]
)

print(results.to_string(index=False))

In [ ]:
leadership_disagreements = comparison[
    comparison["Leadership_context"].astype("string")
    != comparison["Leadership_context_AI"].astype("string")
]

valence_disagreements = comparison[
    comparison["Information_valence"].astype("string")
    != comparison["Information_valence_AI"].astype("string")
]

print("Leadership Context disagreements:",
      len(leadership_disagreements))

print("Information Valence disagreements:",
      len(valence_disagreements))

In [ ]:
results.to_excel("cohens_kappa_results.xlsx", index=False)

comparison.to_excel(
    "coding_comparison_full.xlsx",
    index=False
)

print("Reliability results and coding comparison exported.")

To assess the consistency of the manual coding procedure, ChatGPT (GPT-6) independently classified a stratified random sample of approximately 20% of the dataset for each company using the same predefined coding framework as the researcher. The AI model received the coding guidelines and the relevant textual and contextual information but did not have access to the researcher's classifications. Agreement between the researcher and the AI model was assessed using Cohen's κ (Cohen, 1960), calculated separately for Leadership Context and Information Valence. Unweighted Cohen's κ was used because the primary analysis focused on exact agreement between categorical classifications, treating all disagreements equally. Although Information Valence has an ordinal structure, it was treated as a nominal variable for the primary reliability assessment.

The analysis yielded a Cohen's κ of [INSERT VALUE] for Leadership Context and [INSERT VALUE] for Information Valence, with exact agreement rates of [INSERT PERCENTAGE]% and [INSERT PERCENTAGE]%, respectively, based on [INSERT N] and [INSERT N] valid paired classifications. These results indicate [INTERPRET THE OBSERVED LEVEL OF AGREEMENT].